In [0]:


%sql
CREATE SCHEMA IF NOT EXISTS medical_pipeline.silver;

In [0]:
from pyspark.sql.functions import regexp_replace , col , unix_timestamp
proc_df = spark.table("medical_pipeline.bronze.procedures")
proc_df= proc_df.select('start',
 'stop',
 'patient',
 'encounter_id',
 'code',
 'description',
 'base_cost',
 'reasoncode',
 'reasondescription')

proc_df = proc_df.withColumn(
    "base_cost",
    regexp_replace("base_cost", ",", "").cast("double")
)
proc_df = proc_df.fillna({
    "reasondescription": "unknown",
    "description": "unknown"
})
proc_df = proc_df.dropDuplicates(["patient", "encounter_id", "start", "code"])
proc_df = proc_df.filter(
    col("patient").isNotNull() &
    col("encounter_id").isNotNull() &
    col("start").isNotNull()
)

proc_df = proc_df.withColumn(
    "duration_minutes",
    (unix_timestamp("stop") - unix_timestamp("start")) / 60
)



In [0]:


proc_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_pipeline.silver.procedures")

In [0]:
%sql
select * from medical_pipeline.silver.procedures;

In [0]:
%sql
select * from medical_pipeline.silver.encounters_silver;

In [0]:
%sql
-- Create gold layer schema for dimensional model
CREATE SCHEMA IF NOT EXISTS medical_pipeline.gold;

In [0]:
%sql
-- Dimension Table: Patient Information
-- Type: SCD Type 2 (tracks historical changes)
CREATE OR REPLACE TABLE medical_pipeline.gold.dim_patient (
  patient_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  patient_id STRING NOT NULL,                        -- Natural key from source
  first_name STRING,
  last_name STRING,
  gender STRING,
  birth_date DATE,
  race STRING,
  ethnicity STRING,
  city STRING,
  state STRING,
  county STRING,
  zip STRING,
  
  -- SCD Type 2 columns
  effective_date DATE NOT NULL,
  end_date DATE,
  is_current BOOLEAN NOT NULL DEFAULT TRUE,
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_dim_patient PRIMARY KEY (patient_key)
) USING DELTA
COMMENT 'Patient dimension table with SCD Type 2 for tracking historical changes';

In [0]:
%sql
-- Dimension Table: Procedure Types
-- Type: SCD Type 1 (overwrite changes)
CREATE OR REPLACE TABLE medical_pipeline.gold.dim_procedure_type (
  procedure_type_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  procedure_code DOUBLE NOT NULL,                          -- Natural key
  procedure_description STRING NOT NULL,
  procedure_category STRING,                                -- Grouping/category
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_dim_procedure_type PRIMARY KEY (procedure_type_key),
  CONSTRAINT uk_procedure_code UNIQUE (procedure_code)
) USING DELTA
COMMENT 'Procedure type dimension - lookup for procedure codes and descriptions';

In [0]:
%sql
-- Dimension Table: Encounter Types
-- Type: SCD Type 1 (overwrite changes)
CREATE OR REPLACE TABLE medical_pipeline.gold.dim_encounter_type (
  encounter_type_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  encounter_code BIGINT NOT NULL,                          -- Natural key
  encounter_description STRING NOT NULL,
  encounter_class STRING,                                   -- ambulatory, emergency, inpatient, etc.
  encounter_category STRING,                                -- Grouping/category
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_dim_encounter_type PRIMARY KEY (encounter_type_key),
  CONSTRAINT uk_encounter_code UNIQUE (encounter_code)
) USING DELTA
COMMENT 'Encounter type dimension - lookup for encounter codes, descriptions, and classes';

In [0]:
%sql
-- Dimension Table: Insurance Payers
-- Type: SCD Type 2 (tracks historical changes)
CREATE OR REPLACE TABLE medical_pipeline.gold.dim_payer (
  payer_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  payer_id STRING NOT NULL,                       -- Natural key from source
  payer_name STRING,
  payer_type STRING,                              -- Medicare, Medicaid, Commercial, etc.
  payer_address STRING,
  payer_city STRING,
  payer_state STRING,
  payer_zip STRING,
  
  -- SCD Type 2 columns
  effective_date DATE NOT NULL,
  end_date DATE,
  is_current BOOLEAN NOT NULL DEFAULT TRUE,
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_dim_payer PRIMARY KEY (payer_key)
) USING DELTA
COMMENT 'Payer (insurance) dimension with SCD Type 2 for tracking historical changes';

In [0]:
%sql
-- Dimension Table: Reason Codes
-- Type: SCD Type 1 (overwrite changes)
CREATE OR REPLACE TABLE medical_pipeline.gold.dim_reason (
  reason_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  reason_code DOUBLE NOT NULL,                     -- Natural key
  reason_description STRING NOT NULL,
  reason_category STRING,                           -- Grouping/category
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_dim_reason PRIMARY KEY (reason_key),
  CONSTRAINT uk_reason_code UNIQUE (reason_code)
) USING DELTA
COMMENT 'Reason code dimension - lookup for medical reason codes and descriptions';

In [0]:
%sql
-- Dimension Table: Date Dimension
-- Type: Static reference table
CREATE OR REPLACE TABLE medical_pipeline.gold.dim_date (
  date_key INT NOT NULL,                     -- YYYYMMDD format (e.g., 20260326)
  full_date DATE NOT NULL,
  day_of_week INT,                           -- 1=Sunday, 7=Saturday
  day_name STRING,                           -- Monday, Tuesday, etc.
  day_of_month INT,
  day_of_year INT,
  week_of_year INT,
  month INT,
  month_name STRING,
  quarter INT,
  year INT,
  is_weekend BOOLEAN,
  is_holiday BOOLEAN,
  fiscal_year INT,
  fiscal_quarter INT,
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
) USING DELTA
COMMENT 'Date dimension for time-based analysis and reporting';

In [0]:
%sql
-- Fact Table: Procedures
-- Grain: One row per procedure performed
CREATE OR REPLACE TABLE medical_pipeline.gold.fact_procedures (
  procedure_fact_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  
  -- Foreign keys to dimensions
  patient_key BIGINT NOT NULL,
  procedure_type_key BIGINT NOT NULL,
  encounter_key BIGINT NOT NULL,
  reason_key BIGINT,
  start_date_key INT NOT NULL,
  stop_date_key INT,
  
  -- Degenerate dimensions (transaction identifiers kept in fact)
  encounter_id STRING NOT NULL,
  
  -- Date/Time stamps (for precise querying)
  start_timestamp TIMESTAMP NOT NULL,
  stop_timestamp TIMESTAMP,
  
  -- Measures (numeric facts)
  base_cost DOUBLE,
  duration_minutes DOUBLE,
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_fact_procedures PRIMARY KEY (procedure_fact_key)
) USING DELTA
PARTITIONED BY (start_date_key)
COMMENT 'Fact table for procedure events with measures and dimensional references';

In [0]:
%sql
-- Fact Table: Encounters
-- Grain: One row per patient encounter
CREATE OR REPLACE TABLE medical_pipeline.gold.fact_encounters (
  encounter_fact_key BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  
  -- Foreign keys to dimensions
  patient_key BIGINT NOT NULL,
  encounter_type_key BIGINT NOT NULL,
  payer_key BIGINT NOT NULL,
  reason_key BIGINT,
  start_date_key INT NOT NULL,
  stop_date_key INT,
  
  -- Degenerate dimensions (transaction identifiers kept in fact)
  encounter_id STRING NOT NULL,
  
  -- Date/Time stamps (for precise querying)
  start_timestamp TIMESTAMP NOT NULL,
  stop_timestamp TIMESTAMP,
  
  -- Measures (numeric facts)
  base_encounter_cost DOUBLE,
  total_claim_cost DOUBLE,
  payer_coverage DOUBLE,
  patient_responsibility DOUBLE,                           -- Derived: total - coverage
  duration_hours DOUBLE,                                    -- Derived: stop - start
  
  -- Audit columns
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  
  -- Constraints
  CONSTRAINT pk_fact_encounters PRIMARY KEY (encounter_fact_key),
  CONSTRAINT uk_encounter_id UNIQUE (encounter_id)
) USING DELTA
PARTITIONED BY (start_date_key)
COMMENT 'Fact table for encounter events with cost measures and dimensional references';

# Medical Data Warehouse - Dimensional Model

## 🏗️ Architecture Overview
This dimensional model follows a **Star Schema** design pattern for optimal query performance and business user accessibility.

---

## 📊 Dimension Tables (6)

### 1. **dim_patient** (SCD Type 2)
* **Purpose**: Patient demographics and attributes
* **Key**: `patient_key` (surrogate), `patient_id` (natural)
* **Tracks**: Historical changes to patient information
* **Attributes**: Name, gender, birth date, race, ethnicity, location

### 2. **dim_procedure_type** (SCD Type 1)
* **Purpose**: Procedure code lookup
* **Key**: `procedure_type_key` (surrogate), `procedure_code` (natural)
* **Attributes**: Code, description, category

### 3. **dim_encounter_type** (SCD Type 1)
* **Purpose**: Encounter type lookup
* **Key**: `encounter_type_key` (surrogate), `encounter_code` (natural)
* **Attributes**: Code, description, class (ambulatory/emergency/inpatient)

### 4. **dim_payer** (SCD Type 2)
* **Purpose**: Insurance payer information
* **Key**: `payer_key` (surrogate), `payer_id` (natural)
* **Tracks**: Historical changes to payer details
* **Attributes**: Name, type, address

### 5. **dim_reason** (SCD Type 1)
* **Purpose**: Medical reason code lookup
* **Key**: `reason_key` (surrogate), `reason_code` (natural)
* **Attributes**: Code, description, category

### 6. **dim_date** (Static)
* **Purpose**: Time-based analysis
* **Key**: `date_key` (YYYYMMDD format)
* **Attributes**: All date parts (day, week, month, quarter, year, fiscal periods)

---

## 📈 Fact Tables (2)

### 1. **fact_procedures**
* **Grain**: One row per procedure performed
* **Partitioned by**: `start_date_key`
* **Foreign Keys**: patient_key, procedure_type_key, encounter_key, reason_key, start_date_key, stop_date_key
* **Measures**: `base_cost`, `duration_minutes`
* **Degenerate Dimension**: `encounter_id`

### 2. **fact_encounters**
* **Grain**: One row per patient encounter
* **Partitioned by**: `start_date_key`
* **Foreign Keys**: patient_key, encounter_type_key, payer_key, reason_key, start_date_key, stop_date_key
* **Measures**: `base_encounter_cost`, `total_claim_cost`, `payer_coverage`, `patient_responsibility`, `duration_hours`
* **Degenerate Dimension**: `encounter_id`

---

## 🔗 Relationships

```
                    dim_date
                       |
                   (date_key)
                       |
        +--------------+---------------+
        |                              |
   fact_procedures              fact_encounters
        |                              |
        +------+-------+-------+-------+
               |       |       |       |
          dim_patient  |   dim_payer  |
           dim_procedure_type    dim_encounter_type
                  dim_reason
```

---

## 🎯 Design Patterns Used

1. **Surrogate Keys**: Auto-incrementing BIGINT keys for all dimensions
2. **SCD Type 2**: For patient and payer dimensions (tracks history)
3. **SCD Type 1**: For lookup tables (overwrites changes)
4. **Degenerate Dimensions**: Transaction IDs stored in fact tables
5. **Date Dimension**: Enables flexible time-based analysis
6. **Partitioning**: Facts partitioned by date for query performance
7. **Audit Columns**: Created/updated timestamps on all tables

---

## 💡 Usage Examples

### Total procedures cost by patient:
```sql
SELECT 
  p.first_name, p.last_name,
  COUNT(*) as procedure_count,
  SUM(f.base_cost) as total_cost
FROM fact_procedures f
JOIN dim_patient p ON f.patient_key = p.patient_key
WHERE p.is_current = TRUE
GROUP BY p.first_name, p.last_name;
```

### Monthly encounter costs by payer:
```sql
SELECT 
  d.year, d.month_name,
  py.payer_name,
  SUM(f.total_claim_cost) as total_claims
FROM fact_encounters f
JOIN dim_date d ON f.start_date_key = d.date_key
JOIN dim_payer py ON f.payer_key = py.payer_key
WHERE py.is_current = TRUE
GROUP BY d.year, d.month_name, py.payer_name;
```